# 05 - Domain Feature Engineering

## 1. Domain Feature Engineering Rationale
Raw features capture isolated customer traits. By constructing targeted financial and behavioral ratios, we help tree-based models capture non-linear relationships with fewer splits:

1. **`Income_Per_Family`**:
   $$\text{Income\_Per\_Family} = \frac{\text{Income}}{\text{Family}}$$
   Captures per-capita disposable income.

2. **`CCAvg_to_Income_Ratio`**:
   $$\text{CCAvg\_to\_Income\_Ratio} = \frac{\text{CCAvg} \times 12}{\text{Income}}$$
   Annual credit card spending proportional to annual income.

3. **`Mortgage_to_Income_Ratio`**:
   $$\text{Mortgage\_to\_Income\_Ratio} = \frac{\text{Mortgage}}{\text{Income}}$$
   Debt burden indicator.

4. **`Has_Mortgage`**:
   Binary indicator: $1$ if $\text{Mortgage} > 0$, else $0$.

5. **`Total_Accounts`**:
   Count of banking relationships: $\text{Securities Account} + \text{CD Account} + \text{Online} + \text{CreditCard}$.

6. **`Experience_to_Age_Ratio`**:
   Career velocity proxy: $\frac{\text{Experience}}{\text{Age}}$.

7. **`High_Networth_Indicator`**:
   Affluent flag: $\text{Income} \ge 100\text{k}$ and $\text{CCAvg} \ge 3\text{k}$.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.features.feature_engineering import LoanFeatureEngineer

df = pd.read_csv(project_root / "data" / "interim" / "cleaned_bank_loan.csv")
print("Interim data loaded. Shape:", df.shape)

Interim data loaded. Shape: (5000, 14)


## 2. Applying the Sklearn-Compatible LoanFeatureEngineer Transformer

In [2]:
engineer = LoanFeatureEngineer(drop_identifiers=True)
df_features = engineer.fit_transform(df)

print("Transformed DataFrame Shape:", df_features.shape)
print("\nEngineered Feature Columns:")
display(df_features.head())

Transformed DataFrame Shape: (5000, 19)

Engineered Feature Columns:


,Age,Experience,Income,Family,CCAvg,Education,Mortgage,Personal Loan,Securities Account,CD Account,Online,CreditCard,Income_Per_Family,CCAvg_to_Income_Ratio,Mortgage_to_Income_Ratio,Has_Mortgage,Total_Accounts,Experience_to_Age_Ratio,High_Networth_Indicator
0,25,1,49,4,1.6,1,0,0,1,0,0,0,12.249997,0.391837,0.0,0,1,0.040000,0
1,45,19,34,3,1.5,1,0,0,1,0,0,0,11.333330,0.529412,0.0,0,1,0.422222,0
2,39,15,11,1,1.0,1,0,0,0,0,0,0,10.999989,1.090909,0.0,0,0,0.384615,0
3,35,9,100,1,2.7,2,0,0,0,0,0,0,99.999900,0.324000,0.0,0,0,0.257143,0
4,35,8,45,4,1.0,2,0,0,0,0,0,1,11.249997,0.266667,0.0,0,1,0.228571,0


## 3. Correlation of Engineered Features with Target

In [3]:
corr_with_target = df_features.corr()["Personal Loan"].sort_values(ascending=False)
print("Correlation with Personal Loan (Engineered + Base Features):")
display(corr_with_target.to_frame(name="Correlation"))

Correlation with Personal Loan (Engineered + Base Features):


,Correlation
Personal Loan,1.000000
Income,0.502462
High_Networth_Indicator,0.445650
CCAvg,0.366889
CD Account,0.316355
Income_Per_Family,0.241725
Mortgage,0.142095
Education,0.136722
Total_Accounts,0.099957
Family,0.061367


## 4. Summary
- All engineered features successfully created without data leakage.
- `Income_Per_Family`, `High_Networth_Indicator`, and `CCAvg_to_Income_Ratio` show notable correlation with loan acceptance.